# BTC Data Pull via Massive API
Pulls full BTC/USD daily data and saves to `data/btc_daily.csv`.
Only run once to populate the data file.

In [11]:
import requests
import pandas as pd
import os
from datetime import date
from urllib.parse import quote

API_KEY = '7gjThzOOPPal8jKWBbDl8lgp6ZM1DSYh'
BASE_URL = 'https://api.massive.com'
os.makedirs('data', exist_ok=True)

In [12]:
TICKER = 'X:BTCUSD'
# start from 2016 as the market was much thinner previously
FROM_DATE = '2016-01-01'
TO_DATE = str(date.today())

ticker_encoded = quote(TICKER, safe='')
url = f'{BASE_URL}/v2/aggs/ticker/{ticker_encoded}/range/1/day/{FROM_DATE}/{TO_DATE}'
params = {
    'adjusted': 'true',
    'sort': 'asc',
    'limit': 50000,
    'apiKey': API_KEY
}

all_results = []

while url:
    resp = requests.get(url, params=params)
    print(f'Status: {resp.status_code}')

    data = resp.json()

    results = data.get('results', [])
    all_results.extend(results)

    url = data.get('next_url')
    params = {}

print(f'\nTotal rows fetched: {len(all_results)}')

Status: 200

Total rows fetched: 3894


In [13]:
df = pd.DataFrame(all_results)

# Rename columns 
df = df.rename(columns={
    't': 'timestamp_ms',
    'o': 'open',
    'h': 'high',
    'l': 'low',
    'c': 'close',
    'v': 'volume',
    'vw': 'vwap',
    'n': 'num_trades'
})

# Convert ms timestamp to date
df['date'] = pd.to_datetime(df['timestamp_ms'], unit='ms').dt.date
df = df[['date', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'num_trades']]
df = df.sort_values('date').reset_index(drop=True)

df.to_csv('data/btc_daily.csv', index=False)
df.head()

,date,open,high,low,close,volume,vwap,num_trades
0,2016-01-01,434.17899,434.96999,430.74000,431.50531,14.348580,433.1974,13
1,2016-01-02,434.00000,434.99000,430.32476,434.98000,26.759482,432.9899,169
2,2016-01-03,434.49000,434.49000,422.98000,426.57001,23.611030,427.4345,34
3,2016-01-04,427.77001,433.96000,426.56000,431.08415,40.875985,430.6320,60
4,2016-01-05,431.06001,434.72990,429.00000,431.32000,34.082065,430.5136,73


Simple Data checks

In [14]:
print('=== Shape ===')
print(df.shape)

print('\n=== Date range ===')
print(f'From : {df["date"].min()}')
print(f'To   : {df["date"].max()}')

print('\n=== Missing values ===')
print(df.isnull().sum())

=== Shape ===
(3894, 8)

=== Date range ===
From : 2016-01-01
To   : 2026-08-29

=== Missing values ===
date          0
open          0
high          0
low           0
close         0
volume        0
vwap          0
num_trades    0
dtype: int64


In [15]:
# summary stats
df.describe()

,open,high,low,close,volume,vwap,num_trades
count,3894.000000,3894.000000,3894.000000,3894.000000,3894.000000,3894.000000,3.894000e+03
mean,32833.531277,33558.341205,32062.028208,32852.583676,39201.090623,32821.338530,3.907171e+05
std,32734.522930,33293.365098,32143.298159,32740.553741,41840.952531,32723.080505,3.192636e+05
min,364.480000,373.597320,0.060000,364.490000,14.348580,370.761600,1.300000e+01
25%,6624.305000,6804.875000,6431.675000,6609.297500,11009.419639,6613.045200,1.584862e+05
50%,20231.535000,20783.450000,19764.000000,20235.400000,26434.775852,20234.517900,3.241605e+05
75%,57357.307500,58609.875000,55619.125000,57362.367500,54210.763675,57181.103250,5.604858e+05
max,124765.900000,126296.000000,123115.770000,124720.090000,528732.390968,124887.519800,3.224462e+06


In [16]:
# Check for any missing days in subsetted range
df['date'] = pd.to_datetime(df['date'])
full_range = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='D')
missing_dates = full_range.difference(df['date'])

print(f'Missing dates: {len(missing_dates)}')

Missing dates: 0


In [17]:
df.tail()

,date,open,high,low,close,volume,vwap,num_trades
3889,2026-08-25,78981.59,81300.0,77832.00,78526.80,15749.198615,79473.7385,1657545
3890,2026-08-26,78509.50,79270.0,77627.00,79008.60,2837.918108,78478.7295,142903
3891,2026-08-27,79026.18,80848.4,78551.99,80275.34,17313.903566,79917.1690,928325
3892,2026-08-28,80275.35,81500.0,76845.71,77839.19,22525.332426,78885.2195,1041980
3893,2026-08-29,77841.80,77968.0,77345.28,77706.64,2707.983833,77639.9814,271982


In [19]:
weekly = df.resample('W', on='date').size().rename('daily_obs')
weeks_with_data = (weekly > 0).sum()
total_daily_obs = len(df)

print(f'Total daily rows: {total_daily_obs}')
print(f'Total weeks: {len(weekly)}')

Total daily rows: 3894
Total weeks: 557
